In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# 1. 선샌니가 보고 계신 PC 주소로 변경!
url = "https://cafe.naver.com/projectiofficial" 

# 2. 크롬 브라우저인 척 위장
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 3. 데이터 요청
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

try:
    #  멤버수 (캡처 화면을 바탕으로 제가 미리 짰습니다!)
    member_element = soup.select_one('.mem-cnt-info em') 
    member_text = member_element.text if member_element else "0"

    #  전체글보기 (선샌니가 Copy selector 한 값을 아래 따옴표 안에 붙여넣어주세요!)
    # 예시: post_element = soup.select_one('#cafe-info-data > ul > li:nth-child(4) > em')
    post_element = soup.select_one('#cafe-menu > div.box-g-m > ul:nth-child(1) > li:nth-child(1)') 
    post_text = post_element.text if post_element else "0"

    # 숫자만 깔끔하게 빼기
    member_count = int(re.sub(r'[^0-9]', '', member_text))
    total_posts = int(re.sub(r'[^0-9]', '', post_text))

    print(f" 카페 가입자 수: {member_count:,}명")
    print(f" 전체 게시글 수: {total_posts:,}개")

except Exception as e:
    print(f"데이터 추출 실패  오류 내용: {e}")

✅ 카페 가입자 수: 25,279명
✅ 전체 게시글 수: 127,955개


In [7]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://cafe.naver.com/projectiofficial" 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

try:
    # 1. 가입자 수 (성공했던 완벽한 코드 그대로!)
    member_element = soup.select_one('.mem-cnt-info em')
    member_count = int(re.sub(r'[^0-9]', '', member_element.text)) if member_element else 0
    print(f"✅ 카페 가입자 수: {member_count:,}명")

    # 2. 게시글 수 (진짜 찐 불도저 방식 🚜)
    total_posts = 0
    
    # 🌟 핵심: 특정 구역을 찾지 않고, 페이지 내의 모든 <li> 태그를 싹 다 가져옵니다.
    all_li_tags = soup.find_all('li') 
    
    for li in all_li_tags:
        # 공백과 줄바꿈을 다 지우고 깔끔한 텍스트만 봅니다.
        text_content = li.text.strip().replace('\n', '').replace(' ', '')
        
        # 만약 그 줄에 '전체글'이라는 단어가 포함되어 있다면?!
        if '전체글' in text_content:
            # 글자는 무시하고 숫자(콤마 포함)만 모조리 긁어옵니다.
            numbers = re.findall(r'[\d,]+', text_content)
            
            if numbers:
                # 찾은 숫자 덩어리 중 첫 번째 값을 가져와서 콤마 떼고 숫자로 변환!
                total_posts = int(numbers[0].replace(',', ''))
                break # 찾았으니 반복문 즉시 종료!

    print(f"✅ 전체 게시글 수: {total_posts:,}개")

except Exception as e:
    print(f"오류 발생: {e}")

✅ 카페 가입자 수: 25,279명
✅ 전체 게시글 수: 127,957개


In [8]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from concurrent.futures import ThreadPoolExecutor # 병렬 처리를 위한 일꾼 소환!

# 1. 분석할 카페 아이디 리스트 (여기에 왕창 넣으셔도 됩니다!)
cafe_ids = ['projectiofficial', 'tteokbokk1', 'syoka?email=aba19787d0dde87bee8a21ff0ec27e86', 'dunggeure', 'akakang']

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 2. 한 개의 카페를 수집하는 함수 정의 (선샌니가 성공시킨 그 불도저 로직!)
def scrape_cafe(cafe_id):
    url = f"https://cafe.naver.com/{cafe_id}"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 가입자 수
        member_element = soup.select_one('.mem-cnt-info em')
        member_count = int(re.sub(r'[^0-9]', '', member_element.text)) if member_element else 0
        
        # 게시글 수 (불도저 방식)
        total_posts = 0
        all_li_tags = soup.find_all('li')
        for li in all_li_tags:
            text_content = li.text.strip().replace('\n', '').replace(' ', '')
            if '전체글' in text_content:
                numbers = re.findall(r'[\d,]+', text_content)
                if numbers:
                    total_posts = int(numbers[0].replace(',', ''))
                break
        
        print(f"✅ {cafe_id} 수집 완료!")
        return {'카페아이디': cafe_id, '가입자수': member_count, '게시글수': total_posts}
    
    except Exception as e:
        print(f"❌ {cafe_id} 실패: {e}")
        return {'카페아이디': cafe_id, '가입자수': 0, '게시글수': 0}

# 3. 멀티스레드 실행 (동시에 5개씩 드가자!)
print("🏁 멀티스레드 크롤링을 시작합니다...")
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(scrape_cafe, cafe_ids))

# 4. 결과 저장
df = pd.DataFrame(results)
df.to_csv('cime_parallel_data.csv', index=False, encoding='utf-8-sig')

print(f"\n🎉 총 {len(df)}개 카페 수집 완료! 'cime_parallel_data.csv'를 확인해 보세요.")

🏁 멀티스레드 크롤링을 시작합니다...
✅ akakang 수집 완료!
✅ tteokbokk1 수집 완료!
✅ dunggeure 수집 완료!
✅ syoka?email=aba19787d0dde87bee8a21ff0ec27e86 수집 완료!
✅ projectiofficial 수집 완료!

🎉 총 5개 카페 수집 완료! 'cime_parallel_data.csv'를 확인해 보세요.


In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import urllib.parse
import re
import time

# 1. 선샌니의 데이터 프레임
data = {'streamer_name': ['프로젝트아이', '스텔라이브', '이세계아이돌']} 
df = pd.DataFrame(data)

headers = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 10; SM-G981B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.162 Mobile Safari/537.36"
}

# 2. 첫 번째 아이디만 냅다 물어오는 함수!
def get_first_cafe_id(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 검색 결과의 링크들을 순서대로 확인합니다.
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match:
                    cafe_id = match.group(1)
                    
                    # 네이버 잡동사니 링크만 아니면...
                    if len(cafe_id) > 2 and 'search' not in cafe_id:
                        # 💥 핵심: 찾자마자 고민 안 하고 바로 리턴해버림!!! 💥
                        return cafe_id 
                        
        return None
    except Exception as e:
        print(f"오류: {e}")
        return None

print("🏎️ 노빠꾸 직진 크롤러 출동합니다...\n")
found_ids = []

# 3. 데이터프레임 순회하며 바로바로 꽂아넣기
for name in df['streamer_name']:
    cafe_id = get_first_cafe_id(name)
    found_ids.append(cafe_id)
    print(f"🎯 [{name}] ➡️ 1빠따 아이디: {cafe_id if cafe_id else '실패 😭'}")
    time.sleep(1) # 그래도 네이버 눈치는 봐야 하니 1초 휴식

# 4. 결과 저장!
df['cafe_id'] = found_ids
print("\n🎉 수집 완료! 이대로 바로 저장합니다!")
print(df)

# df.to_csv('cime_fast_mapped.csv', index=False, encoding='utf-8-sig')

🏎️ 노빠꾸 직진 크롤러 출동합니다...

🎯 [프로젝트아이] ➡️ 1빠따 아이디: projectiofficial
🎯 [스텔라이브] ➡️ 1빠따 아이디: tteokbokk1
🎯 [이세계아이돌] ➡️ 1빠따 아이디: steamindiegame

🎉 수집 완료! 이대로 바로 저장합니다!
  streamer_name           cafe_id
0        프로젝트아이  projectiofficial
1         스텔라이브        tteokbokk1
2        이세계아이돌    steamindiegame


In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib.parse
import re
import time

# 1. 선샌니의 DB(CSV 파일) 불러오기!
file_path = '샘플_소프트콘.csv'

# 혹시 한글 인코딩 에러가 날까 봐 보험용으로 try-except를 걸어둡니다.
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

# 🌟 핵심 스킬 1: 중복된 스트리머 이름 제거하고 고유한 이름만 리스트로 뽑기!
unique_streamers = df['streamer_name'].dropna().unique()
print(f"📊 총 {len(df)}줄의 데이터에서, {len(unique_streamers)}명의 고유 스트리머를 찾아냈습니다!\n")

headers = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 10; SM-G981B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.162 Mobile Safari/537.36"
}

# (아까 만든 노빠꾸 1빠따 크롤러 함수!)
def get_first_cafe_id(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match:
                    cafe_id = match.group(1)
                    if len(cafe_id) > 2 and 'search' not in cafe_id:
                        return cafe_id 
        return None
    except Exception as e:
        print(f"오류: {e}")
        return None

# 3. 고유 스트리머 이름으로만 검색 시작!
print("🏎️ 노빠꾸 직진 크롤러 출동합니다...\n")
mapping_data = [] # 이름 - 카페아이디 짝꿍을 저장할 바구니

for name in unique_streamers:
    cafe_id = get_first_cafe_id(name)
    mapping_data.append({'streamer_name': name, 'cafe_id': cafe_id})
    print(f"🎯 [{name}] ➡️ 1빠따 아이디: {cafe_id if cafe_id else '실패 😭'}")
    time.sleep(1) # 차단 방지 매너 1초!

# 4. 찾아온 짝꿍 데이터를 데이터프레임으로 변환
mapping_df = pd.DataFrame(mapping_data)

# 🌟 핵심 스킬 2: 원본 DB와 매핑 테이블 합치기 (SQL의 LEFT JOIN과 똑같아요!)
final_df = pd.merge(df, mapping_df, on='streamer_name', how='left')

# 5. 최종 결과 저장!
final_df.to_csv('최종_소프트콘_카페추가.csv', index=False, encoding='utf-8-sig')

print("\n🎉 수집 및 병합 완료!")
print("폴더에 생성된 '최종_소프트콘_카페추가.csv'를 확인해 보세요! 맨 오른쪽에 cafe_id 열이 쫙 붙어있을 겁니다!")
# --- (이전 코드 맨 아래에 이어서 붙여넣기) ---

# 🌟 보너스 스킬: 성공/실패 인원수 집계하기
# mapping_df를 기준으로 고유 스트리머 중 몇 명을 성공/실패했는지 셉니다.
success_count = mapping_df['cafe_id'].notna().sum()
fail_count = mapping_df['cafe_id'].isna().sum()

print("="*40)
print("📊 [크롤링 최종 결과 보고서]")
print(f"✅ 성공 (카페 찾음): {success_count}명")
print(f"❌ 실패 (카페 못찾음): {fail_count}명")
print("="*40)

# 만약 실패한 스트리머 이름만 쏙 뽑아서 보고 싶다면?
failed_streamers = mapping_df[mapping_df['cafe_id'].isna()]['streamer_name'].tolist()
if fail_count > 0:
    print(f"😭 못 찾은 스트리머 명단: {failed_streamers}")

📊 총 34000줄의 데이터에서, 100명의 고유 스트리머를 찾아냈습니다!

🏎️ 노빠꾸 직진 크롤러 출동합니다...

🎯 [강지] ➡️ 1빠따 아이디: tteokbokk1
🎯 [계춘회] ➡️ 1빠따 아이디: 실패 😭
🎯 [고차비] ➡️ 1빠따 아이디: moomoo
🎯 [금사향] ➡️ 1빠따 아이디: tkgidsla
🎯 [남궁혁] ➡️ 1빠따 아이디: 실패 😭
🎯 [네네코 마시로] ➡️ 1빠따 아이디: tteokbokk1
🎯 [뇨롱이] ➡️ 1빠따 아이디: nyorong
🎯 [늦잠] ➡️ 1빠따 아이디: gamstfan
🎯 [달콤레나 씨] ➡️ 1빠따 아이디: 실패 😭
🎯 [독케익] ➡️ 1빠따 아이디: 실패 😭
🎯 [둥그레] ➡️ 1빠따 아이디: bbongshinkmk
🎯 [디디디용] ➡️ 1빠따 아이디: projectiofficial
🎯 [라코코 RAKOKO] ➡️ 1빠따 아이디: rakoko
🎯 [램램1] ➡️ 1빠따 아이디: 실패 😭
🎯 [류시호 RyuSiho] ➡️ 1빠따 아이디: projectiofficial
🎯 [마레 플로스] ➡️ 1빠따 아이디: mareflos
🎯 [마뫄] ➡️ 1빠따 아이디: mamwa
🎯 [망내] ➡️ 1빠따 아이디: projectiofficial
🎯 [맹숙] ➡️ 1빠따 아이디: 실패 😭
🎯 [모라라] ➡️ 1빠따 아이디: 실패 😭
🎯 [모카형] ➡️ 1빠따 아이디: moomoo
🎯 [미녕이데려오깨] ➡️ 1빠따 아이디: givemecsplz
🎯 [방찌] ➡️ 1빠따 아이디: bangjjeefan
🎯 [백곰파] ➡️ 1빠따 아이디: 100gompa
🎯 [부쿠키] ➡️ 1빠따 아이디: moomoo
🎯 [블레어 로즈] ➡️ 1빠따 아이디: projectiofficial
🎯 [빙하유] ➡️ 1빠따 아이디: s2dia
🎯 [사키하네 후야] ➡️ 1빠따 아이디: tteokbokk1
🎯 [시라유키 히나] ➡️ 1빠따 아이디: 실패 😭
🎯 [아라하시 타비] ➡️ 1빠따 아이디: tteokbokk1
🎯 [아리사] ➡️ 1빠따 아이디: v

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib.parse
import re
import time

# --- 기존 코드와 동일 ---
file_path = 'final_streamer_dataset_1p_122p.csv'
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

unique_streamers = df['streamer_name'].dropna().unique()
print(f"📊 총 {len(unique_streamers)}명의 고유 스트리머 수집 시작!\n")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 🌟 업그레이드된 스마트 크롤러: 매니저 신원조회 기능 탑재!
def get_cafe_id_smart(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. 상위 3개 후보 카페 아이디 수집
        candidates = []
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match:
                    cafe_id = match.group(1)
                    if len(cafe_id) > 2 and 'search' not in cafe_id and cafe_id not in candidates:
                        candidates.append(cafe_id)
                    if len(candidates) == 3: # 딱 3개까지만 용의선상에 올립니다.
                        break
                        
        if not candidates:
            return None

        # 2. [플랜 B] 후보 카페들에 직접 들어가서 매니저 이름 확인!
        # 검색어(스트리머명)의 공백을 없애고 소문자로 만들어서 비교 준비
        clean_streamer_name = streamer_name.replace(" ", "").lower()
        
        for cafe_id in candidates:
            cafe_url = f"https://cafe.naver.com/{cafe_id}" # 매니저 확인은 PC주소가 편해요!
            cafe_res = requests.get(cafe_url, headers=headers)
            cafe_soup = BeautifulSoup(cafe_res.text, 'html.parser')
            
            manager_name = ""
            # 이전에 썼던 '불도저' 로직 재등장! 카페 정보란에서 '매니저' 텍스트 찾기
            for li in cafe_soup.select('#cafe-info-data > ul > li'):
                text_content = li.text.strip().replace('\n', '').replace(' ', '')
                if '매니저' in text_content:
                    manager_name = text_content.replace('매니저', '').lower()
                    break
            
            # 매니저 이름에 스트리머 이름이 포함되어 있다면?!
            if clean_streamer_name in manager_name:
                print(f"    ✨ [검거 완료] 매니저({manager_name}) 일치! -> {cafe_id}")
                return cafe_id # 정답 찾았으니 즉시 리턴!

        # 3. 만약 3개 다 매니저 이름이 안 맞으면? 
        # (스트리머가 부계정을 쓰거나 찐팬이 매니저인 경우)
        # -> 기존 80%의 성공률을 지키기 위해 그냥 제일 첫 번째(1빠따) 카페를 던져줍니다.
        return candidates[0]
        
    except Exception as e:
        print(f"오류: {e}")
        return None

# --- 실행 파트 ---
print("🕵️‍♂️ 스마트 탐정단 출동합니다...\n")
mapping_data = []

for name in unique_streamers:
    cafe_id = get_cafe_id_smart(name)
    mapping_data.append({'streamer_name': name, 'cafe_id': cafe_id})
    print(f"🎯 [{name}] ➡️ 최종 선택 아이디: {cafe_id if cafe_id else '실패 😭'}")
    time.sleep(1) # 차단 방지 매너 1초!

# 합치고 저장하는 로직
mapping_df = pd.DataFrame(mapping_data)
final_df = pd.merge(df, mapping_df, on='streamer_name', how='left')
final_df.to_csv('최종_소프트콘_매니저검증판.csv', index=False, encoding='utf-8-sig')

# 결과 보고서
success_count = mapping_df['cafe_id'].notna().sum()
fail_count = mapping_df['cafe_id'].isna().sum()

print("\n" + "="*40)
print("📊 [크롤링 최종 결과 보고서]")
print(f"✅ 성공 (카페 찾음): {success_count}명")
print(f"❌ 실패 (카페 못찾음): {fail_count}명")
print("="*40)

📊 총 11599명의 고유 스트리머 수집 시작!

🕵️‍♂️ 스마트 탐정단 출동합니다...

🎯 [00곤뇽00] ➡️ 최종 선택 아이디: 실패 😭
🎯 [0리엘0] ➡️ 최종 선택 아이디: 실패 😭
🎯 [0무지0] ➡️ 최종 선택 아이디: 실패 😭
🎯 [0서이안0] ➡️ 최종 선택 아이디: 0nepunchk1ng
🎯 [0영식이] ➡️ 최종 선택 아이디: takstudioofficial
🎯 [0우리아0] ➡️ 최종 선택 아이디: khantata
🎯 [1010m] ➡️ 최종 선택 아이디: 실패 😭
🎯 [1김상어1] ➡️ 최종 선택 아이디: 실패 😭
🎯 [4대가호 최하엘] ➡️ 최종 선택 아이디: 실패 😭
🎯 [8Fabe8] ➡️ 최종 선택 아이디: 실패 😭
🎯 [9unD] ➡️ 최종 선택 아이디: 실패 😭
🎯 [ADIA 아디아] ➡️ 최종 선택 아이디: 실패 😭
🎯 [B 059 비오] ➡️ 최종 선택 아이디: 실패 😭
🎯 [BJ쯔코] ➡️ 최종 선택 아이디: clidfan
🎯 [BJ펭귄] ➡️ 최종 선택 아이디: hanwhaeaglesfancafe
🎯 [Bomnun] ➡️ 최종 선택 아이디: 실패 😭
🎯 [breeeezy] ➡️ 최종 선택 아이디: 실패 😭
🎯 [Bulzip] ➡️ 최종 선택 아이디: 실패 😭
🎯 [BunnyBlossom] ➡️ 최종 선택 아이디: 실패 😭
🎯 [chezzly nacho] ➡️ 최종 선택 아이디: 실패 😭
🎯 [Chief하랑] ➡️ 최종 선택 아이디: 실패 😭
🎯 [CHORING초링] ➡️ 최종 선택 아이디: 실패 😭
🎯 [chzzk멋진카렌] ➡️ 최종 선택 아이디: 실패 😭
🎯 [CPPshooter] ➡️ 최종 선택 아이디: 실패 😭
🎯 [DANI님] ➡️ 최종 선택 아이디: gamstfan
🎯 [Dark Hood] ➡️ 최종 선택 아이디: 실패 😭
🎯 [DA쿠아] ➡️ 최종 선택 아이디: 실패 😭
🎯 [DDAK] ➡️ 최종 선택 아이디: 실패 😭
🎯 [DEVILU] ➡️ 최종 선택 아이디: 실패 😭
🎯 [DNen] ➡️ 최종 선택

In [6]:
# 🔍 '쇼 카'님 카페 찾기 성공 여부 확인 코드

target_name = '강지' 

# 1. 매핑 결과(mapping_df)에서 해당 스트리머 찾기
# (참고: mapping_df는 고유 스트리머 이름과 카페ID가 담긴 표입니다)
result = mapping_df[mapping_df['streamer_name'] == target_name]

if not result.empty:
    cafe_id = result.iloc[0]['cafe_id']
    
    if pd.notna(cafe_id):
        print(f"✨ [검거 성공!] '{target_name}'님의 카페 아이디를 찾았습니다!")
        print(f"📌 카페 아이디: {cafe_id}")
        print(f"🔗 바로가기: https://cafe.naver.com/{cafe_id}")
    else:
        print(f"😭 [검거 실패...] '{target_name}'님은 아쉽게도 실패한 11,441명 중 한 분이네요.")
        print(f"💡 이유 추정: 네이버 차단으로 검색 결과가 안 떴거나, 카페를 운영하지 않으실 수 있어요.")
else:
    print(f"❓ '{target_name}'님은 애초에 수집 대상 명단에 없었던 것 같아요. 이름을 다시 확인해 보세요!")

😭 [검거 실패...] '강지'님은 아쉽게도 실패한 11,441명 중 한 분이네요.
💡 이유 추정: 네이버 차단으로 검색 결과가 안 떴거나, 카페를 운영하지 않으실 수 있어요.
